# Exercise 1 — From Pixels to Visual Evidence

**Course:** Computer Vision for Industrial Systems  
**Topic:** imaging fundamentals before machine learning  
**Format:** Binder / Jupyter Notebook

This notebook implements the core message of Lecture 2:

> **Before training a model, make sure the relevant visual signal is visible, sufficiently sampled, stable, and measurable.**

You will work through three parts:

1. **Sampling and feature visibility** — object-space resolution, aliasing, anti-aliasing.
2. **Image formation perturbations** — blur, noise, saturation, bit depth, gamma.
3. **Simple inspection pipeline** — thresholding, morphology, connected components, measurement.

The notebook is intentionally still **pre-machine-learning**. The aim is to understand the image evidence itself.

## How to work with this notebook

You will see code cells with explicit `TODO` sections. Replace the missing parts with your own code.

Difficulty markers:

- 🟢 **Basic** — everyone should be able to finish this during the exercise.
- 🟡 **Intermediate** — requires a little more reasoning or debugging.
- 🔴 **Advanced** — optional challenge if you finish early.

The most important part is not only the code, but the **engineering interpretation**:
What does the result mean for an industrial vision system?

## 0. Setup

Run this cell first. It imports the libraries and helper functions used throughout the notebook.

The helper functions are deliberately simple and located in `src/cvis_utils.py`.

In [ ]:
from pathlib import Path
import sys
import math
import os

# Ensure Matplotlib uses a writable cache directory in Binder/sandbox environments.
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib_cache").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

# Make helper module importable when running from notebooks/
repo_root = Path.cwd().parents[1] if Path.cwd().name == "solutions" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(repo_root / "src"))

from cvis_utils import (
    load_gray, show_images, plot_histograms,
    mm_per_pixel, pixels_per_feature,
    downsample_image, motion_blur_kernel, apply_motion_blur,
    add_gaussian_noise, gamma_correct, quantize_image,
    sobel_energy, laplacian_variance,
    otsu_segment, clean_mask, region_table, iou
)

DATA = repo_root / "data"
SYN = DATA / "synthetic"
IND = DATA / "industrial_like"

print("Repository root:", repo_root)
print("Synthetic data exists:", SYN.exists())
print("Industrial-like data exists:", IND.exists())

## 0.1 Load and inspect images

We use two types of images:

1. **Synthetic image patterns** for controlled sampling and aliasing experiments.
2. **Industrial-style surface images** for defect visibility, thresholding, and measurement.

The industrial-style images are generated teaching images. They are useful for a stable first Binder exercise because the solution is reproducible and lightweight.

In [ ]:
checker = load_gray(SYN / "checkerboard_highres.png")
brick = load_gray(SYN / "brick_wall_synthetic.png")
surface = load_gray(IND / "images" / "surface_defect_bright_scratch_easy.png")
mask_gt = load_gray(IND / "masks" / "surface_defect_bright_scratch_easy_mask.png") > 0.5

show_images(
    [checker, brick, surface, mask_gt],
    ["checkerboard", "brick-like texture", "surface defect", "ground-truth mask"],
    ncols=4,
    figsize=(14, 4),
)

# Part A — Object-space resolution, sampling, and aliasing

Industrial question:

> **Is the relevant feature large enough in pixel space to be detected or measured reliably?**

We use the running design scenario from the lecture:

- field-of-view width: **400 mm**
- horizontal pixel count: **2448 px**
- relevant defect size: **0.5 mm**

You first compute whether this defect is sufficiently sampled. Then you implement downsampling with and without anti-aliasing.

## Task A1 — Compute mm/px and pixels per defect 🟢

The object-space resolution is:

$$
\mathrm{mm/px} = \frac{\mathrm{FoV}_{x}}{N_x}
$$

The number of pixels across a physical feature is:

$$
\mathrm{pixels\ per\ feature}
=
\frac{\mathrm{feature\ size\ in\ mm}}{\mathrm{mm/px}}
$$

### Your task

Implement two short functions:

1. `student_mm_per_pixel(fov_mm, n_pixels)`
2. `student_pixels_per_feature(feature_size_mm, mm_per_px)`

### Engineering interpretation

A feature spanning only 1–2 pixels is usually not reliable.  
A feature spanning around 3 pixels may be detectable under good conditions, but it is fragile.  
For localization or measurement, more pixel support is usually needed.

In [ ]:
def student_mm_per_pixel(fov_mm: float, n_pixels: int) -> float:
    """Return object-space resolution in mm/px.

    TODO:
    1. Check that n_pixels is positive.
    2. Divide the field of view in millimeters by the number of pixels.
    3. Return the result as a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement student_mm_per_pixel")


def student_pixels_per_feature(feature_size_mm: float, mm_per_px: float) -> float:
    """Return how many pixels span a physical feature.

    TODO:
    1. Check that mm_per_px is positive.
    2. Divide the physical feature size by the object-space resolution.
    3. Return the result as a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement student_pixels_per_feature")

In [ ]:
# Basic self-checks. These should pass once you implemented the functions.
assert np.isclose(student_mm_per_pixel(400.0, 2000), 0.2)
assert np.isclose(student_pixels_per_feature(0.5, 0.1), 5.0)
print("A1 self-checks passed.")

In [ ]:
fov_mm = 400.0
n_px = 2448
defect_size_mm = 0.5

mm_px = student_mm_per_pixel(fov_mm, n_px)
px_defect = student_pixels_per_feature(defect_size_mm, mm_px)

print(f"Object-space resolution: {mm_px:.4f} mm/px")
print(f"0.5 mm defect spans: {px_defect:.2f} px")

### Short interpretation A1 🟢

Answer in **2–4 sentences**:

1. Is the 0.5 mm defect sufficiently sampled?
2. Is this more promising for classification, localization, or measurement?
3. Which additional image-quality factors matter besides pixel count?

In [ ]:
answer_A1 = """
Replace this text with your answer.

"""
print(answer_A1)

## Task A2 — Downsampling and aliasing 🟢 / 🟡

When we reduce the resolution of an image, we reduce the sampling frequency.  
If the original image contains fine structures that are too high-frequency for the new sampling grid, those structures do not simply disappear. Instead, they can reappear as **false low-frequency patterns**. This is called **aliasing**.

A simple way to reduce aliasing is:

1. **Low-pass filter** the image first, for example with a Gaussian filter.  
2. Then **subsample** the image, for example by keeping only every `k`-th pixel.

This is called **anti-aliasing**.

---

### Downsampling without anti-aliasing

For an integer downsampling factor `k`, a very simple downsampling operation is:

$$
I_{\mathrm{down}}[i,j] = I[k i, k j]
$$

This keeps only every `k`-th pixel and discards the others.

This is fast, but it can create aliasing because high-frequency details are not removed before sampling.

---

### Downsampling with anti-aliasing

With anti-aliasing, we first smooth the image:

$$
I_{\mathrm{smooth}} = G_{\sigma} * I
$$

Here, `*` denotes convolution, and `G_sigma` is a Gaussian low-pass filter.

A one-dimensional Gaussian kernel is:

$$
G_{\sigma}(x)
=
\frac{1}{\sqrt{2\pi}\,\sigma}
\exp\left(
-\frac{x^2}{2\sigma^2}
\right)
$$

In the notebook, we apply this Gaussian filter once horizontally and once vertically.

Then we downsample the smoothed image:

$$
I_{\mathrm{down}}[i,j] = I_{\mathrm{smooth}}[k i, k j]
$$

---

### Engineering interpretation

- Downsampling **without** anti-aliasing preserves sharpness visually, but can create false structures.
- Downsampling **with** anti-aliasing deliberately smooths the image first, so the result may look softer but is usually more faithful.
- In industrial inspection, aliasing can make fine textures, edges, or defects appear incorrectly.

### Your task A2

Implement your own simple downsampling method.

You must implement three functions:

1. `gaussian_kernel_1d(sigma, radius)`
2. `lowpass_gaussian(image, sigma)`
3. `downsample_manual(image, factor, anti_aliasing)`

Do **not** use `skimage.transform.resize` in this task.  
The goal is to understand what anti-aliasing actually does.

In [ ]:
def gaussian_kernel_1d(sigma: float, radius: int) -> np.ndarray:
    """Create a normalized 1D Gaussian kernel.

    TODO:
    1. Check that sigma is positive.
    2. Create positions x = [-radius, ..., 0, ..., +radius].
    3. Compute weights: exp(-x**2 / (2 * sigma**2)).
    4. Normalize the kernel so that kernel.sum() == 1.
    5. Return the kernel.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement gaussian_kernel_1d")


def lowpass_gaussian(image: np.ndarray, sigma: float) -> np.ndarray:
    """Smooth an image with a separable Gaussian filter.

    TODO:
    1. Choose radius = ceil(3 * sigma).
    2. Create a 1D Gaussian kernel using gaussian_kernel_1d.
    3. Convolve the image along axis 1.
    4. Convolve the result along axis 0.
    5. Return the smoothed image.

    Hint:
    Use scipy.ndimage.convolve1d with mode="reflect".
    This package is already imported as ndi.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement lowpass_gaussian")


def downsample_manual(image: np.ndarray, factor: int, anti_aliasing: bool = False) -> np.ndarray:
    """Downsample an image by an integer factor.

    TODO:
    If anti_aliasing is False:
        - return image[::factor, ::factor]

    If anti_aliasing is True:
        - choose sigma depending on the factor
        - smooth the image with lowpass_gaussian
        - then return smoothed[::factor, ::factor]

    Hint:
        A reasonable starting point is sigma = factor / 2.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement downsample_manual")

In [ ]:
# Test your Gaussian kernel.
kernel = gaussian_kernel_1d(sigma=1.0, radius=3)

print("Kernel:", kernel)
print("Kernel sum:", kernel.sum())

assert kernel.ndim == 1, "The kernel should be one-dimensional."
assert np.isclose(kernel.sum(), 1.0), "The Gaussian kernel should sum to 1."
assert len(kernel) == 7, "With radius=3, the kernel length should be 7."

print("A2 Gaussian kernel test passed.")

In [ ]:
factor = 4

brick_no_aa = downsample_manual(brick, factor=factor, anti_aliasing=False)
brick_with_aa = downsample_manual(brick, factor=factor, anti_aliasing=True)

show_images(
    [brick, brick_no_aa, brick_with_aa],
    [
        "original image",
        f"downsampled by {factor}, no anti-aliasing",
        f"downsampled by {factor}, with anti-aliasing"
    ],
    ncols=3,
    figsize=(14, 4)
)

## Task A3 — Intensity profiles 🟡

Visual inspection is important, but sometimes we want to look more precisely at how pixel values change across an image.

An **intensity profile** is a one-dimensional slice through an image.

For example, if we choose one horizontal image row, we can plot the gray value of each pixel along that row:

$$
p[j] = I[r,j]
$$

where:

- `I` is the image,
- `r` is the selected row index,
- `j` is the column index,
- `p[j]` is the intensity value at column `j` in row `r`.

So instead of looking at the full 2D image, we inspect one line of pixel values.

---

### Why is this useful?

An intensity profile helps us see:

- where edges occur,
- how strong the contrast is,
- whether fine structures are still visible,
- whether downsampling has changed the apparent signal,
- whether aliasing has introduced false patterns.

In this task, we use intensity profiles to compare:

1. the original image,
2. the downsampled image **without** anti-aliasing,
3. the downsampled image **with** anti-aliasing.

---

### Important detail: different x-axes

The downsampled images have fewer pixels than the original image.

For example:

- original image: 512 pixels wide
- downsampled image by factor 4: 128 pixels wide

If we plot the downsampled profile directly, it would appear shorter.  
To compare it fairly with the original image, we rescale the x-axis so that all profiles cover the same original image width.

That means the y-values come from the downsampled image, but the x-axis is mapped back to the original coordinate system.

---

### What you should look for

When you compare the profiles, ask:

- Do the peaks and valleys still occur at similar positions?
- Does the profile without anti-aliasing show false oscillations?
- Does the anti-aliased profile look smoother?
- Which profile better represents the original structure?
- Which profile might mislead a later inspection algorithm?

The goal is not only to plot lines, but to understand how sampling changes the visual evidence.

In [ ]:
row_original = brick.shape[0] // 2
row_down = brick_no_aa.shape[0] // 2

plt.figure(figsize=(12, 4))

# TODO:
# 1. Plot the original row: brick[row_original, :]
# 2. Plot the downsampled row without anti-aliasing.
#    Use np.linspace(0, brick.shape[1] - 1, brick_no_aa.shape[1]) as x-axis.
# 3. Plot the downsampled row with anti-aliasing.
# 4. Add labels so that the legend is meaningful.
# YOUR CODE HERE
raise NotImplementedError("Plot the three intensity profiles")

plt.title("Intensity profiles before and after downsampling")
plt.xlabel("horizontal position")
plt.ylabel("intensity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Short interpretation A2/A3 🟢 / 🟡

Answer in **3–5 sentences**:

1. What visual differences do you observe between the downsampled image with and without anti-aliasing?
2. Which version looks sharper?
3. Which version is more faithful to the original structure?
4. Why can the sharper-looking version be misleading?
5. What could this mean for industrial inspection of fine textures, edges, or small defects?

In [ ]:
answer_A2 = """
Replace this text with your answer.

"""
print(answer_A2)

# Part B — Blur, noise, saturation, bit depth, and gamma

Industrial question:

> **How do image-formation perturbations change the evidence available to a downstream algorithm?**

We now simulate common effects from Lecture 2:

- motion blur,
- noise,
- saturation / clipping,
- reduced bit depth,
- gamma / display processing.

The goal is not to build a perfect physical camera simulator. The goal is to observe how visual evidence changes.

## Task B1 — Motion blur and edge energy 🟢

When an object moves during the exposure time of the camera, its image is smeared along the direction of motion.  
This is called **motion blur**.

Motion blur is especially problematic in industrial computer vision when we need sharp edges, small defects, readable text, or precise localization.

---

### Motion blur estimate from the lecture

In the lecture, we used the following approximate blur estimate:

$$
\mathrm{blur}_{\mathrm{px}}
\approx
\frac{v_{\mathrm{obj}} \cdot t_{\mathrm{exp}}}{\mathrm{mm/px}}
$$

where:

| Symbol | Meaning | Typical unit |
|---|---|---|
| $$\mathrm{blur}_{\mathrm{px}}$$ | motion blur measured in pixels | pixels |
| $$v_{\mathrm{obj}}$$ | object speed in the real world | mm/s |
| $$t_{\mathrm{exp}}$$ | exposure time of the camera | s |
| $$\mathrm{mm/px}$$ | object-space resolution | mm/px |

The numerator

$$
v_{\mathrm{obj}} \cdot t_{\mathrm{exp}}
$$

is the physical distance the object moves during the exposure.

Dividing by

$$
\mathrm{mm/px}
$$

converts this physical motion into motion blur measured in pixels.

---

### Example interpretation

If an object moves by `0.4 mm` during the exposure and the object-space resolution is `0.1 mm/px`, then:

$$
\mathrm{blur}_{\mathrm{px}}
=
\frac{0.4\ \mathrm{mm}}{0.1\ \mathrm{mm/px}}
=
4\ \mathrm{px}
$$

So the object is blurred by approximately **4 pixels**.

For small defects or sharp edges, this can already be a serious problem.

---

### What we do in this notebook

In this task, we do not simulate the full physical camera process.  
Instead, we apply a simple **linear motion-blur kernel** to an image.

A motion-blur kernel averages pixels along one direction, which approximates the effect of motion during exposure.

After applying blur, we compute two simple sharpness indicators:

| Indicator | What it measures | Interpretation |
|---|---|---|
| **Sobel energy** | average edge strength | lower values often mean weaker edges |
| **Laplacian variance** | local high-frequency variation | lower values often mean less sharpness / more blur |

These are not perfect quality metrics, but they are useful for seeing how blur reduces image evidence.

---

### Engineering interpretation

In industrial inspection, motion blur can make:

- edges less sharp,
- small defects less visible,
- segmentation masks less accurate,
- OCR/text harder to read,
- measurements less reliable.

The goal of this task is to see that blur is not just a visual effect.  
It changes the information available to any later algorithm.

In [ ]:
# Apply motion blur to the `surface` image.

# TODO:
# 1. Use apply_motion_blur(surface, length=..., angle_deg=...).
# 2. Try length=21 and angle_deg=0 first.
surface_blur = ...


show_images(
    [surface, surface_blur],
    ["original surface defect", "motion-blurred"],
    ncols=2,
    figsize=(10, 4),
)

print("Sobel energy original:", sobel_energy(surface))
print("Sobel energy blurred :", sobel_energy(surface_blur))
print("Laplacian variance original:", laplacian_variance(surface))
print("Laplacian variance blurred :", laplacian_variance(surface_blur))

## Task B2 — Noise, saturation, and bit depth 🟢 / 🟡

Create three modified versions of the surface image:

1. **noisy image** — simulates sensor noise or high gain,
2. **clipped image** — simulates saturation / overexposure,
3. **quantized image** — simulates reduced bit depth.

Then compare both the images and their histograms.

Practical reminder:

- Noise makes weak evidence unstable.
- Saturation destroys information in clipped regions.
- Quantization reduces the number of possible intensity levels.

In [ ]:
# Create noisy, clipped, and quantized versions.

# TODO:
# 1. surface_noisy: use add_gaussian_noise(surface, sigma=..., seed=...)
# 2. surface_clipped: multiply the image by a gain and clip to [0, 1]
# 3. surface_quantized: use quantize_image(surface, bits=...)
#
# Suggested starting values:
# sigma = 0.08
# gain = 1.7
# bits = 3
surface_noisy = ...
surface_clipped = ...
surface_quantized = ...


show_images(
    [surface, surface_noisy, surface_clipped, surface_quantized],
    ["original", "noisy", "clipped / saturated", "quantized"],
    ncols=4,
    figsize=(14, 4),
)

plot_histograms(
    [surface, surface_noisy, surface_clipped, surface_quantized],
    ["original", "noisy", "clipped", "quantized"],
    bins=64,
    figsize=(14, 3),
)

## Task B3 — Gamma correction and measurement meaning 🟡

Gamma correction is often useful for **display**, but it changes the relationship between physical signal and pixel value.

A common gamma transform is:

$$
I_{\mathrm{out}} = I_{\mathrm{in}}^{1/\gamma}
$$

where:

| Symbol | Meaning | Typical range / unit |
|---|---|---|
| $$I_{\mathrm{in}}$$ | input image intensity before gamma correction | usually normalized to $$[0,1]$$ |
| $$I_{\mathrm{out}}$$ | output image intensity after gamma correction | usually normalized to $$[0,1]$$ |
| $$\gamma$$ | gamma value controlling the nonlinear mapping | dimensionless |

---

### What does the transform do?

If $$\gamma > 1$$, then:

$$
1/\gamma < 1
$$

and the transform tends to make darker intensities brighter.

For example, with $$\gamma = 2.2$$:

$$
I_{\mathrm{out}} = I_{\mathrm{in}}^{1/2.2}
$$

This is useful for display because human brightness perception is nonlinear.

---

### Why can this be dangerous for measurement?

For measurement tasks, we often want pixel values to remain physically meaningful.

For example, if the image is approximately linear, then a pixel value of `0.6` corresponds to roughly twice the signal of a pixel value of `0.3`.

After gamma correction, this is no longer true.

So gamma correction can be dangerous if it is applied without documentation, because it changes:

- intensity ratios,
- contrast relationships,
- histogram shape,
- threshold behavior,
- measurement assumptions.

---

### Engineering interpretation

Gamma correction is not “wrong”.  
But you must know whether you are working with:

- a **linear image** for measurement, or
- a **display-processed image** for visualization.

In industrial computer vision, undocumented display processing can hide important assumptions.

In [ ]:
def gamma_correct_manual(image, gamma: float):
    """
    Apply gamma correction to an image.

    The gamma transform is:

        I_out = I_in ** (1 / gamma)

    Parameters
    ----------
    image:
        2D grayscale image. Pixel values should be in [0, 1].

    gamma:
        Gamma value. Must be positive.
        Example: gamma=2.2 is common for display-like correction.

    Returns
    -------
    corrected:
        Gamma-corrected image, clipped to [0, 1].

    TODO:
    1. Check that gamma is positive.
    2. Clip the input image to the range [0, 1].
    3. Apply the transform image ** (1 / gamma).
    4. Return the corrected image.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement gamma_correct_manual")

In [ ]:
# Test your gamma correction implementation.

test_values = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
test_gamma = 2.2

test_corrected = gamma_correct_manual(test_values, gamma=test_gamma)

print("Input values:     ", test_values)
print("Gamma corrected:  ", test_corrected)

assert np.isclose(test_corrected[0], 0.0), "0 should stay 0."
assert np.isclose(test_corrected[-1], 1.0), "1 should stay 1."
assert np.all(test_corrected >= 0) and np.all(test_corrected <= 1), "Values should stay in [0, 1]."

print("Gamma correction test passed.")

In [ ]:
gamma = 2.2

# Apply your own gamma correction implementation.

surface_gamma = gamma_correct_manual(
    surface,
    gamma=gamma
)

show_images(
    [surface, surface_gamma],
    ["original / linear-like", f"gamma corrected, gamma={gamma}"],
    ncols=2,
    figsize=(10, 4)
)

plot_histograms(
    [surface, surface_gamma],
    ["original", "gamma corrected"],
    bins=64,
    figsize=(10, 3)
)

### Short interpretation B3

Answer in 3–5 sentences:

1. What visual change do you observe after gamma correction?
2. How did the histogram change?
3. Does gamma correction preserve a linear relationship between physical signal and pixel value?
4. Why could undocumented gamma correction be problematic for measurement or thresholding?
5. In which situation would gamma correction still be useful?

In [ ]:
answer_B = """
Replace this text with your answer.

"""
print(answer_B)

# Part C — Thresholding, morphology, and simple measurement

Industrial question:

> **Can a simple non-ML pipeline isolate and measure the defect?**

We build a minimal inspection pipeline:

1. threshold the image,
2. clean the binary mask,
3. measure connected components,
4. compare against the known synthetic mask.

This is not a production-ready algorithm. It is a diagnostic exercise:  
if even a simple method cannot find clean visual evidence, the acquisition setup may need improvement before machine learning.

## Task C1 — Otsu thresholding 🟢 / 🟡

We now want to segment the bright scratch from the darker background.

A simple segmentation rule is:

$$
\mathrm{mask}[i,j] =
\begin{cases}
1, & I[i,j] > T \\
0, & I[i,j] \leq T
\end{cases}
$$

where:

| Symbol | Meaning |
|---|---|
| $$I[i,j]$$ | intensity value of the image at pixel position $$(i,j)$$ |
| $$T$$ | threshold value |
| $$\mathrm{mask}[i,j]$$ | binary mask value at pixel position $$(i,j)$$ |

The question is:

> How do we choose a good threshold automatically?

---

### Idea of Otsu thresholding

Otsu's method assumes that the image histogram contains two dominant groups:

1. background pixels
2. foreground pixels

It tries to find the threshold that separates these two groups as well as possible.

For each possible threshold $$T$$, the pixels are split into two classes:

- class 0: pixels with intensity $$\leq T$$
- class 1: pixels with intensity $$> T$$

For each threshold, Otsu's method computes how far apart the two class means are, weighted by the class probabilities.

The between-class variance is:

$$
\sigma_B^2(T)
=
w_0(T)\,w_1(T)\,
\left(\mu_0(T)-\mu_1(T)\right)^2
$$

where:

| Symbol | Meaning |
|---|---|
| $$w_0(T)$$ | fraction of pixels in class 0 |
| $$w_1(T)$$ | fraction of pixels in class 1 |
| $$\mu_0(T)$$ | mean intensity of class 0 |
| $$\mu_1(T)$$ | mean intensity of class 1 |
| $$\sigma_B^2(T)$$ | between-class variance for threshold $$T$$ |

Otsu's method chooses the threshold that maximizes this value:

$$
T^* =
\arg\max_T \sigma_B^2(T)
$$

---

### Your task

Implement a simplified version of Otsu thresholding yourself.

You should:

1. compute a histogram of the image,
2. test all possible threshold values,
3. compute the between-class variance for each threshold,
4. choose the threshold with the maximum between-class variance,
5. create a binary mask.

For this specific image, the scratch is **brighter** than the background, so the foreground rule is:

$$
\mathrm{mask} = I > T^*
$$

In [ ]:
def otsu_threshold_manual(image, nbins: int = 256):
    """
    Compute Otsu's threshold manually for a grayscale image.

    Parameters
    ----------
    image:
        2D grayscale image with values in [0, 1].

    nbins:
        Number of histogram bins.

    Returns
    -------
    threshold:
        Threshold value selected by Otsu's method.

    Notes
    -----
    We implement the idea:

        maximize w0 * w1 * (mu0 - mu1)**2

    over all possible histogram thresholds.

    TODO:
    1. Clip the image to [0, 1].
    2. Compute a histogram with `nbins` bins in the range [0, 1].
    3. Compute the bin centers.
    4. Normalize the histogram so that it represents probabilities.
    5. Compute cumulative class probabilities w0 and w1.
    6. Compute cumulative class means mu0 and mu1.
    7. Compute between-class variance.
    8. Return the bin center with the maximum between-class variance.

    Hint:
        Use np.histogram, np.cumsum, and np.argmax.

    Important:
        Avoid division by zero when a class is empty.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement otsu_threshold_manual")


def threshold_image(image, threshold: float, foreground: str = "bright"):
    """
    Convert a grayscale image into a binary mask.

    Parameters
    ----------
    image:
        2D grayscale image.

    threshold:
        Intensity threshold.

    foreground:
        Either "bright" or "dark".

        If foreground == "bright":
            foreground pixels are image > threshold.

        If foreground == "dark":
            foreground pixels are image < threshold.

    Returns
    -------
    mask:
        Boolean binary mask.

    TODO:
    1. If foreground is "bright", return image > threshold.
    2. If foreground is "dark", return image < threshold.
    3. Raise a ValueError for any other value.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement threshold_image")

In [ ]:
# Test your manual Otsu implementation on a simple artificial example.

test_image = np.concatenate([
    np.full(100, 0.2),
    np.full(100, 0.8)
]).reshape(20, 10)

test_threshold = otsu_threshold_manual(test_image, nbins=256)
print("Test threshold:", test_threshold)

assert 0.2 < test_threshold < 0.8, "Threshold should lie between the two intensity groups."

test_mask = threshold_image(test_image, test_threshold, foreground="bright")

assert test_mask.dtype == bool, "Mask should be boolean."
assert test_mask.sum() == 100, "The bright half should be segmented."

print("Manual Otsu test passed.")

In [ ]:
# Segment the bright scratch with your own Otsu implementation.

threshold_value = otsu_threshold_manual(
    surface,
    nbins=256
)

mask_raw = threshold_image(
    surface,
    threshold=threshold_value,
    foreground="bright"
)

print("Otsu threshold:", threshold_value)

show_images(
    [surface, mask_raw],
    ["surface image", "raw threshold mask"],
    ncols=2,
    figsize=(10, 4)
)

In [ ]:
# Visualize the histogram and the selected threshold.

plt.figure(figsize=(8, 4))

plt.hist(
    surface.ravel(),
    bins=64,
    range=(0, 1),
    alpha=0.8
)

plt.axvline(
    threshold_value,
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"Otsu threshold = {threshold_value:.3f}"
)

plt.xlabel("intensity")
plt.ylabel("pixel count")
plt.title("Image histogram with Otsu threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Task C2 — Morphological cleanup 🟢 / 🟡

Thresholding often produces binary masks that are not yet usable for measurement.

Typical problems are:

- small false-positive regions,
- small holes,
- small gaps in the detected object,
- fragmented defect regions.

In this task, we use **mathematical morphology** to clean a binary mask.

---

### Connected-component filtering

A binary mask may contain several connected regions.

A **connected component** is a group of foreground pixels that touch each other.

We can remove very small components by keeping only components whose area is at least a chosen threshold:

$$
\mathrm{area}(C_k) \geq A_{\min}
$$

where:

| Symbol | Meaning |
|---|---|
| $$C_k$$ | connected component number $$k$$ |
| $$\mathrm{area}(C_k)$$ | number of pixels in component $$C_k$$ |
| $$A_{\min}$$ | minimum allowed component area |

This removes small false-positive detections.

---

### Morphological closing

To close small gaps, we use **morphological closing**.

Closing is defined as:

$$
A \bullet B = (A \oplus B) \ominus B
$$

where:

| Symbol | Meaning |
|---|---|
| $$A$$ | binary mask |
| $$B$$ | structuring element, for example a small disk |
| $$\oplus$$ | dilation |
| $$\ominus$$ | erosion |
| $$A \bullet B$$ | closing of mask $$A$$ by structuring element $$B$$ |

Interpretation:

1. **Dilation** expands the foreground and can close small gaps.
2. **Erosion** shrinks the foreground back after dilation.

So closing can connect small breaks without strongly changing the object size.

---

### Intersection-over-union

To compare a predicted mask with the known ground-truth mask, we use **intersection-over-union**:

$$
\mathrm{IoU}
=
\frac{|M_{\mathrm{pred}} \cap M_{\mathrm{gt}}|}
{|M_{\mathrm{pred}} \cup M_{\mathrm{gt}}|}
$$

where:

| Symbol | Meaning |
|---|---|
| $$M_{\mathrm{pred}}$$ | predicted binary mask |
| $$M_{\mathrm{gt}}$$ | ground-truth binary mask |
| $$\cap$$ | intersection: pixels that are foreground in both masks |
| $$\cup$$ | union: pixels that are foreground in at least one mask |

IoU is 1 if the masks match perfectly and 0 if they do not overlap at all.

---

### Your task

Implement the cleanup yourself.

You should:

1. remove small connected components,
2. apply morphological closing,
3. compute IoU before and after cleanup,
4. explain whether morphology improved the mask.

For this exercise, we intentionally create a noisy raw mask so that the effect of morphology is visible.

In [ ]:
def create_noisy_mask_for_morphology(mask, seed: int = 42):
    """
    Create a deliberately degraded mask for the morphology task.

    The mask contains:
    - the original main defect,
    - small false-positive blobs,
    - artificial gaps in the defect.

    This is only for teaching morphology.
    """
    rng = np.random.default_rng(seed)

    degraded = mask.astype(bool).copy()
    h, w = degraded.shape

    # Add small false-positive blobs
    for _ in range(18):
        r = rng.integers(2, 5)
        cy = rng.integers(r, h - r)
        cx = rng.integers(r, w - r)

        rr, cc = np.ogrid[:h, :w]
        blob = (rr - cy) ** 2 + (cc - cx) ** 2 <= r ** 2
        degraded[blob] = True

    # Add one or two gaps in the scratch region
    foreground_pixels = np.argwhere(degraded)

    if len(foreground_pixels) > 0:
        for q in [0.40, 0.65]:
            idx = int(q * len(foreground_pixels))
            cy, cx = foreground_pixels[idx]

            gap_radius = 7
            rr, cc = np.ogrid[:h, :w]
            gap = (rr - cy) ** 2 + (cc - cx) ** 2 <= gap_radius ** 2
            degraded[gap] = False

    return degraded


mask_for_cleanup = create_noisy_mask_for_morphology(mask_raw, seed=42)

show_images(
    [mask_raw, mask_for_cleanup, mask_gt],
    ["original raw mask", "noisy mask for morphology task", "ground truth mask"],
    ncols=3,
    figsize=(12, 4)
)

In [ ]:
from skimage import measure, morphology

def remove_small_components_manual(mask, min_size: int):
    """
    Remove connected components smaller than min_size.

    Parameters
    ----------
    mask:
        Boolean binary mask.

    min_size:
        Minimum number of pixels required for a component to be kept.

    Returns
    -------
    filtered:
        Boolean mask containing only components with area >= min_size.

    TODO:
    1. Label connected components using measure.label.
    2. Create an empty boolean output mask.
    3. Loop over all connected components.
    4. For each component, compute its area in pixels.
    5. Keep the component only if area >= min_size.
    6. Return the filtered mask.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement remove_small_components_manual")


def binary_closing_manual(mask, radius: int):
    """
    Apply morphological closing to a binary mask.

    Closing means:

        dilation followed by erosion

    Parameters
    ----------
    mask:
        Boolean binary mask.

    radius:
        Radius of the disk-shaped structuring element.

    Returns
    -------
    closed:
        Boolean mask after morphological closing.

    TODO:
    1. Create a disk-shaped structuring element using morphology.disk(radius).
    2. Apply binary dilation.
    3. Apply binary erosion to the dilated mask.
    4. Return the result.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement binary_closing_manual")


def iou_manual(pred_mask, gt_mask):
    """
    Compute intersection-over-union between two binary masks.

    Parameters
    ----------
    pred_mask:
        Predicted binary mask.

    gt_mask:
        Ground-truth binary mask.

    Returns
    -------
    iou:
        Intersection-over-union score.

    TODO:
    1. Convert both masks to boolean arrays.
    2. Compute the intersection.
    3. Compute the union.
    4. Return intersection / union.
    5. If the union is zero, return 1.0 only if both masks are empty.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement iou_manual")

In [ ]:
# Test remove_small_components_manual on a small artificial mask.

test_mask = np.zeros((10, 10), dtype=bool)
test_mask[1:3, 1:3] = True      # area 4
test_mask[5:9, 5:9] = True      # area 16

test_filtered = remove_small_components_manual(test_mask, min_size=10)

assert test_filtered.sum() == 16, "Only the large component should remain."

# Test IoU manually.
a = np.array([[1, 1, 0],
              [0, 1, 0]], dtype=bool)

b = np.array([[1, 0, 0],
              [0, 1, 1]], dtype=bool)

# intersection = 2, union = 4
assert np.isclose(iou_manual(a, b), 2 / 4), "IoU calculation is incorrect."

print("Morphology and IoU tests passed.")

In [ ]:
min_size = 80
closing_radius = 3

mask_removed_small = remove_small_components_manual(
    mask_for_cleanup,
    min_size=min_size
)

mask_clean = binary_closing_manual(
    mask_removed_small,
    radius=closing_radius
)

show_images(
    [mask_for_cleanup, mask_removed_small, mask_clean, mask_gt],
    [
        "noisy raw mask",
        f"after removing small components\nmin_size={min_size}",
        f"after closing\nradius={closing_radius}",
        "ground truth mask"
    ],
    ncols=4,
    figsize=(14, 4)
)

print("IoU noisy raw mask:", iou_manual(mask_for_cleanup, mask_gt))
print("IoU cleaned mask  :", iou_manual(mask_clean, mask_gt))

### Short interpretation C2

Answer in 3–5 sentences:

1. Which artifacts were removed by connected-component filtering?
2. Did morphological closing reconnect gaps in the scratch?
3. Did the IoU improve after cleanup?
4. Can morphology also make a mask worse?
5. What does this imply for using morphology in an industrial inspection pipeline?

## Task C3 — Region measurement 🟢 / 🟡

Now measure the connected regions in the cleaned mask.

We use the `mm_px` value from Part A to convert area from pixels² to mm².

The result is not a calibrated real measurement, because our images are generated teaching data.  
However, the workflow mirrors a real inspection pipeline:

> image → mask → connected components → measured properties

In [ ]:
# Compute a region table.

# TODO:
# Use region_table(mask_clean, mm_per_px=mm_px).
regions = ...


df_regions = pd.DataFrame(regions)
df_regions

## Task C4 — Try a harder image 🔴

Repeat thresholding and measurement for a harder image.

We use:

- `surface_defect_dark_scratch_01.png` — dark scratch instead of bright scratch

Think carefully:

1. Is the defect brighter or darker than the background?
2. Does the same thresholding strategy still work?
3. Does morphology help or hurt?

This image is intentionally harder. It contains a dark scratch on a structured background with strong vertical texture.

Global Otsu thresholding is expected to produce many false positives here.
The goal is not to obtain a perfect mask, but to diagnose why the simple pipeline fails.

In [ ]:
hard_image_path = IND / "images" / "surface_defect_dark_scratch_01.png"
hard_mask_path = IND / "masks" / "surface_defect_dark_scratch_01_mask.png"

hard_img = load_gray(hard_image_path)
hard_gt = load_gray(hard_mask_path) > 0.5

# Segment and clean the harder image.

# TODO:
# Choose foreground="bright" or foreground="dark".
# Then clean the resulting mask.
hard_raw, hard_thr = ...
hard_clean = ...


show_images(
    [hard_img, hard_raw, hard_clean, hard_gt],
    ["hard image", "raw mask", "clean mask", "ground truth"],
    ncols=4,
    figsize=(14, 4)
)

print("Hard image threshold:", hard_thr)
print("Hard image IoU:", iou(hard_clean, hard_gt))

### Short interpretation C4

Answer in 4–6 sentences:

1. Which parts of the background are incorrectly segmented as defect?
2. Why does global Otsu thresholding fail on this image?
3. Why does morphology not remove the false positives?
4. What image-acquisition change could make this easier?
5. What algorithmic change could make this easier?

## Task C5 — Background correction for a failure case 🔴

In the previous task, global Otsu thresholding failed on the harder image.

The reason is not that Otsu is “wrong” in general.  
The reason is that the image violates one of Otsu's main assumptions:

> foreground and background should be separable mainly by intensity.

In the hard image, the scratch is dark, but the background also contains dark vertical texture.  
Therefore, global thresholding detects many background structures as false positives.

Morphological cleanup alone cannot solve this, because the false-positive background structures are not just tiny isolated pixels. They are large connected regions.

---

### Idea: estimate and remove the slowly varying background

The scratch is a thin dark structure.  
The background texture changes more slowly over space.

We can therefore estimate a smoothed background image:

$$
B = G_{\sigma} * I
$$

where:

| Symbol | Meaning |
|---|---|
| $$I$$ | original hard image |
| $$G_{\sigma}$$ | Gaussian smoothing filter |
| $$\sigma$$ | smoothing strength |
| $$B$$ | estimated slowly varying background |

Because the scratch is darker than the background, we compute a residual image:

$$
R = B - I
$$

where:

| Symbol | Meaning |
|---|---|
| $$R$$ | residual image |
| $$B$$ | estimated background |
| $$I$$ | original image |

Dark thin structures become bright in this residual image.

We then clip negative values:

$$
R_{\mathrm{clip}} = \max(R, 0)
$$

and normalize the residual for thresholding:

$$
R_{\mathrm{norm}} =
\frac{R_{\mathrm{clip}}}
{\max(R_{\mathrm{clip}}) + \epsilon}
$$

where $$\epsilon$$ is a small number to avoid division by zero.

---

### Your task

Implement a simple background-correction pipeline:

1. estimate the background using Gaussian smoothing,
2. compute the dark-scratch residual,
3. clip negative values,
4. normalize the residual,
5. threshold the residual with your manual Otsu method,
6. clean the resulting mask with your morphology functions,
7. compare the IoU to the original global-Otsu result.

This is still not a production-ready defect detector.  
The goal is to understand that **preprocessing must match the image-formation problem**.

In [ ]:
from scipy import ndimage as ndi

def background_correct_dark_features(image, sigma: float):
    """
    Enhance dark thin structures by subtracting the image from a smoothed background.

    Parameters
    ----------
    image:
        2D grayscale image with values in [0, 1].

    sigma:
        Standard deviation of the Gaussian filter used to estimate the background.
        Larger sigma means a smoother background estimate.

    Returns
    -------
    background:
        Smoothed estimate of the slowly varying background.

    residual_norm:
        Normalized residual image in [0, 1].
        Dark thin structures in the original image should appear bright here.

    TODO:
    1. Estimate the background using ndi.gaussian_filter(image, sigma=sigma).
    2. Compute residual = background - image.
       This is useful because the scratch is dark.
    3. Clip negative residual values to 0.
    4. Normalize the residual by dividing by residual.max() + 1e-12.
    5. Return background and residual_norm.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement background_correct_dark_features")

In [ ]:
# Test background correction on a small artificial example.

test = np.ones((50, 50), dtype=float) * 0.6
test[25, 10:40] = 0.2  # dark horizontal line

test_background, test_residual = background_correct_dark_features(
    test,
    sigma=5
)

assert test_background.shape == test.shape, "Background must have same shape as input."
assert test_residual.shape == test.shape, "Residual must have same shape as input."
assert test_residual.min() >= 0, "Residual should be non-negative."
assert test_residual.max() <= 1.0 + 1e-12, "Residual should be normalized to [0, 1]."

show_images(
    [test, test_background, test_residual],
    ["test image", "estimated background", "residual"],
    ncols=3,
    figsize=(10, 3)
)

print("Background-correction test passed.")

In [ ]:
sigma_background = 12

background, residual_norm = background_correct_dark_features(
    hard_img,
    sigma=sigma_background
)

# Threshold the residual image.
# In the residual, the dark scratch should now appear bright.
res_thr = otsu_threshold_manual(
    residual_norm,
    nbins=256
)

res_raw = threshold_image(
    residual_norm,
    threshold=res_thr,
    foreground="bright"
)

# Clean residual-based mask.
res_removed_small = remove_small_components_manual(
    res_raw,
    min_size=80
)

res_clean = binary_closing_manual(
    res_removed_small,
    radius=2
)

show_images(
    [hard_img, background, residual_norm, res_raw, res_clean, hard_gt],
    [
        "hard image",
        f"estimated background\nsigma={sigma_background}",
        "dark-scratch residual",
        "raw residual mask",
        "clean residual mask",
        "ground truth"
    ],
    ncols=6,
    figsize=(18, 4)
)

print("Residual threshold:", res_thr)
print("IoU original global Otsu:", iou_manual(hard_clean, hard_gt))
print("IoU residual method      :", iou_manual(res_clean, hard_gt))

### Short interpretation C5

Answer in 4–6 sentences:

1. Why did global Otsu thresholding fail on the original hard image?
2. What does the Gaussian-smoothed background represent?
3. Why do we compute `background - image` for a dark scratch?
4. Did the residual-based method improve the IoU?
5. What could go wrong if `sigma` is too small or too large?
6. What does this example teach us about preprocessing before machine learning?

# Part D — Homography-based rectification and planar measurement 🟢 / 🟡

So far, we worked with images where the object plane was already approximately fronto-parallel.

In real industrial inspection, this is not always true.  
A camera may observe a flat surface from an oblique angle. Then the same physical object can appear distorted by perspective.

This is a problem if we want to measure:

- distances,
- widths,
- areas,
- object positions,
- defect locations.

A **homography** is a projective transformation that maps points from one image plane to another image plane.

For a planar scene, we can write:

$$
s
\begin{bmatrix}
u' \\
v' \\
1
\end{bmatrix}
=
H
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
$$

where:

| Symbol | Meaning |
|---|---|
| $$u, v$$ | pixel coordinates in the original warped image |
| $$u', v'$$ | pixel coordinates in the rectified image |
| $$H$$ | $$3 \times 3$$ homography matrix |
| $$s$$ | arbitrary scale factor in homogeneous coordinates |

---

## Why this matters

If the object lies on a plane, homography rectification can transform an oblique view into a top-down view.

This makes later measurement more meaningful.

However, a homography only works correctly under important assumptions:

- the inspected surface is approximately planar,
- the point correspondences are correct,
- lens distortion is negligible or already corrected,
- the physical scale in the rectified image is known.

In this task, you will rectify a synthetic planar inspection target and measure a simple object in the rectified coordinate system.

In [ ]:
# Part D setup: load warped planar target and metadata.

import json
import cv2
from skimage import io, img_as_float

HOMO = DATA / "advanced" / "homography"

warped = img_as_float(io.imread(HOMO / "planar_target_warped.png"))
rectified_reference = img_as_float(io.imread(HOMO / "planar_target_rectified.png"))

with open(HOMO / "homography_metadata.json", "r") as f:
    hmeta = json.load(f)

# Source points: fiducials in the warped image.
src_warped = np.array(
    list(hmeta["fiducials_warped_px"].values()),
    dtype=np.float32
)

# Destination points: where those fiducials should be in the rectified image.
dst_rectified = np.array(
    list(hmeta["fiducials_rectified_px"].values()),
    dtype=np.float32
)

output_w, output_h = hmeta["rectified_size_px"]

show_images(
    [warped],
    ["warped original image"],
    ncols=1,
    figsize=(8, 5)
)

print("Warped image shape:", warped.shape)
print("Output rectified size:", output_w, "x", output_h)
print("Source points in warped image:")
print(src_warped)
print("Destination points in rectified image:")
print(dst_rectified)

## Task D1 — Inspect the point correspondences 🟢

A homography is estimated from corresponding points.

We need pairs of points:

$$
(u_i, v_i)
\longleftrightarrow
(u_i', v_i')
$$

where:

| Symbol | Meaning |
|---|---|
| $$(u_i, v_i)$$ | point $$i$$ in the warped image |
| $$(u_i', v_i')$$ | corresponding point $$i$$ in the rectified image |

A homography has 8 degrees of freedom, because the matrix is defined only up to scale.  
Therefore, we need at least **four point correspondences**.

In this notebook, the four corner fiducials are already provided in the metadata.

Your first task is to visualize these points on the warped image.

In [ ]:
def draw_points(image, points, labels=None, color=(1.0, 0.0, 0.0), radius=8):
    """
    Draw points on a copy of an RGB image.

    Parameters
    ----------
    image:
        Input image. Can be grayscale or RGB, values in [0, 1].

    points:
        Array of shape (N, 2), with points in (x, y) order.

    labels:
        Optional list of point labels.

    color:
        RGB color tuple with values in [0, 1].

    radius:
        Radius of the drawn point marker.

    Returns
    -------
    image_points:
        RGB image with points drawn.

    TODO:
    1. Convert grayscale image to RGB if needed.
    2. Make a copy of the image.
    3. For each point (x, y), draw a small filled disk or square.
    4. If labels are given, add text labels using matplotlib later or ignore labels here.
    5. Return the image with drawn points.

    Hint:
        You can use simple array indexing:
        image_points[y-radius:y+radius, x-radius:x+radius, :] = color
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement draw_points")

In [ ]:
# Test/application cell:
point_labels = ["P1", "P2", "P3", "P4"]

warped_with_points = draw_points(
    warped,
    src_warped,
    labels=point_labels,
    color=(1.0, 0.0, 0.0),
    radius=7
)

show_images(
    [warped_with_points],
    ["warped image with fiducial points"],
    ncols=1,
    figsize=(8, 5)
)

## Task D2 — Estimate the homography and rectify the image 🟢 / 🟡

Now we estimate a homography matrix $$H$$ from the point correspondences.

The homography maps points from the warped image into the rectified image:

$$
s
\begin{bmatrix}
u' \\
v' \\
1
\end{bmatrix}
=
H
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
$$

In this basic exercise, you do **not** need to implement the full Direct Linear Transform algorithm manually.  
Instead, use OpenCV:

```python
H, status = cv2.findHomography(src_points, dst_points)

Then use:
rectified = cv2.warpPerspective(image, H, output_size)

where output_size is (output_w, output_h).

In [ ]:
## Student code cell 3 — estimate and warp

def rectify_with_homography(image, src_points, dst_points, output_size):
    """
    Estimate a homography and rectify an image.

    Parameters
    ----------
    image:
        Input warped image.

    src_points:
        Points in the warped image, shape (N, 2).

    dst_points:
        Corresponding points in the rectified image, shape (N, 2).

    output_size:
        Tuple (width, height) for the rectified image.

    Returns
    -------
    rectified:
        Rectified image.

    H:
        Estimated 3x3 homography matrix.

    TODO:
    1. Use cv2.findHomography(src_points, dst_points) to estimate H.
    2. Use cv2.warpPerspective(image, H, output_size) to rectify the image.
    3. Return rectified and H.

    Important:
        OpenCV expects output_size as (width, height), not (height, width).
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement rectify_with_homography")

In [ ]:
rectified, H = rectify_with_homography(
    warped,
    src_points=src_warped,
    dst_points=dst_rectified,
    output_size=(output_w, output_h)
)

print("Estimated homography matrix H:")
print(H)

show_images(
    [warped, rectified, rectified_reference],
    ["warped image", "rectified by you", "reference rectified image"],
    ncols=3,
    figsize=(14, 5)
)

## Task D3 — Check whether the homography maps points correctly 🟡

A good first sanity check is to apply the estimated homography to the source points and see whether they land near the expected destination points.

To apply a homography to a point, we use homogeneous coordinates:

$$
\tilde{x}
=
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
$$

Then:

$$
\tilde{x}' = H \tilde{x}
$$

After multiplication, we convert back to ordinary pixel coordinates:

$$
u' = \frac{\tilde{x}'_1}{\tilde{x}'_3}
$$

$$
v' = \frac{\tilde{x}'_2}{\tilde{x}'_3}
$$

Your task is to implement this point transformation.

In [ ]:
def apply_homography_to_points(points, H):
    """
    Apply a homography to 2D points.

    Parameters
    ----------
    points:
        Array of shape (N, 2), with points in (x, y) order.

    H:
        3x3 homography matrix.

    Returns
    -------
    mapped_points:
        Array of shape (N, 2), transformed points in (x, y) order.

    TODO:
    1. Convert points to homogeneous coordinates by appending a column of ones.
    2. Multiply by H.
       Hint: use matrix multiplication.
    3. Convert back from homogeneous coordinates by dividing by the third coordinate.
    4. Return only the first two coordinates.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement apply_homography_to_points")

In [ ]:
mapped_points = apply_homography_to_points(src_warped, H)

df_points = pd.DataFrame({
    "src_x": src_warped[:, 0],
    "src_y": src_warped[:, 1],
    "mapped_x": mapped_points[:, 0],
    "mapped_y": mapped_points[:, 1],
    "target_x": dst_rectified[:, 0],
    "target_y": dst_rectified[:, 1],
})

df_points["error_px"] = np.sqrt(
    (df_points["mapped_x"] - df_points["target_x"])**2
    +
    (df_points["mapped_y"] - df_points["target_y"])**2
)

df_points

## Task D4 — Measure an object after rectification 🟡
 
The metadata contains a rectangle around a measurement object in the rectified image.
 
Because the rectified image is assumed to have a known object-space scale, we can convert from pixels to millimeters.

The metadata gives the variable:

    mm_per_px_rectified

This value tells us how many millimeters correspond to one pixel in the rectified image.

The physical size is then computed as:

    width_mm  = width_px  * mm_per_px_rectified
    height_mm = height_px * mm_per_px_rectified

Variable meanings:

    width_px
        object width in pixels

    height_px
        object height in pixels

    mm_per_px_rectified
        object-space resolution in the rectified image

    width_mm
        physical object width in millimeters

    height_mm
        physical object height in millimeters

Your task is to compute the physical width and height of the object.

In [ ]:

## Student code cell 5 — measurement

mm_per_px_rectified = hmeta["assumed_mm_per_px_rectified"]
x1, y1, x2, y2 = hmeta["measurement_object_rectified_px"]

# TODO:
# 1. Compute width_px = x2 - x1
# 2. Compute height_px = y2 - y1
# 3. Convert both to millimeters using mm_per_px_rectified

width_px = ...
height_px = ...

width_mm = ...
height_mm = ...

print(f"Object width:  {width_px:.1f} px = {width_mm:.2f} mm")
print(f"Object height: {height_px:.1f} px = {height_mm:.2f} mm")

In [ ]:
rectified_with_box = rectified.copy()

# Draw the measurement rectangle.
# We work with a simple copy and array indexing for a visible box.
rectified_with_box[int(y1):int(y2), int(x1):int(x1)+3, :] = [1, 0, 0]
rectified_with_box[int(y1):int(y2), int(x2)-3:int(x2), :] = [1, 0, 0]
rectified_with_box[int(y1):int(y1)+3, int(x1):int(x2), :] = [1, 0, 0]
rectified_with_box[int(y2)-3:int(y2), int(x1):int(x2), :] = [1, 0, 0]

show_images(
    [rectified, rectified_with_box],
    ["rectified image", "measurement rectangle"],
    ncols=2,
    figsize=(12, 5)
)

### Short interpretation D

Answer in 4–6 sentences:

1. What visual difference do you observe between the warped and rectified image?
2. Why is measurement on the warped image risky?
3. What assumptions are required for homography-based rectification to be valid?
4. What does the homography correct?
5. What does it **not** correct?
6. Why is this relevant for industrial computer vision?

# Final engineering interpretation

Write a short engineering recommendation.

Imagine you are the engineer responsible for this inspection setup.  
Before using a machine-learning model, what would you check or improve?

Your answer should refer to:

1. sampling / resolution,
2. blur / noise / saturation / gamma,
3. thresholding / segmentation,
4. imaging setup recommendation before ML.

In [ ]:
final_recommendation = """
1. Sampling / resolution:


2. Blur / noise / saturation / gamma:


3. Thresholding / segmentation:


4. Imaging setup recommendation before ML:


"""
print(final_recommendation)

## Optional extension ideas

If you finish early:

1. Repeat A2 with different downsampling factors.
2. Repeat A2 with different Gaussian widths `sigma`.
3. Try different motion-blur lengths and plot sharpness metrics.
4. Compare IoU for multiple defect images.
5. Try adaptive thresholding or edge detection.
6. Estimate the maximum exposure time for a chosen blur tolerance.

The key question remains:

> **Which imaging changes would make the visual evidence more reliable before using ML?**